In [ ]:
import numpy as np # linear algebra
from google.colab import drive
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, applications, callbacks
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import Markdown
import kagglehub

drive.mount('/content/drive')

Mounted at /content/drive


## K-mean clustering (Demo)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.cluster import KMeans

# 1. Load the image (BGR) and convert to RGB
image_path = "/content/drive/MyDrive/tomato_dataset/segmented/Tomato___Late_blight/1ea54b98-1f87-4da7-9bc2-f6460f18b376___RS_Late.B 6713_final_masked.jpg"
img_bgr = cv2.imread(image_path)
if img_bgr is None:
    raise ValueError(f"Could not read {image_path}. Please check the path.")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# 2. Convert to Lab color space
img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)

# 3. Create a mask to remove the black background
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
_, mask = cv2.threshold(gray, 5, 255, cv2.THRESH_BINARY)

# 4. Extract only the leaf region from Lab image using the mask
lab_pixels = img_lab[mask > 0]
# Use only the a and b channels for clustering (ignoring lightness)
ab_pixels = lab_pixels[:, 1:3].astype(np.float32)

# 5. K-means clustering on a and b channels
k = 2  # Change this value if you want more clusters.
kmeans = KMeans(n_clusters=k, random_state=0, n_init=10)
kmeans.fit(ab_pixels)
cluster_labels = kmeans.labels_
centers = kmeans.cluster_centers_

# 6. Map the cluster labels back to the full image dimensions
full_labels = np.full(mask.shape, -1, dtype=int)
full_labels[mask > 0] = cluster_labels

# 7. Create a composite segmented Lab image using cluster centroids
seg_img_lab = img_lab.copy()
for i in range(k):
    # Create a mask for the current cluster
    cluster_mask = (full_labels == i)
    # Replace a and b channels with the cluster centroid values
    seg_img_lab[cluster_mask, 1] = centers[i, 0]
    seg_img_lab[cluster_mask, 2] = centers[i, 1]

# 8. Convert composite segmented image back to RGB for display
seg_img_rgb = cv2.cvtColor(seg_img_lab, cv2.COLOR_LAB2RGB)
segmented_pil = Image.fromarray(seg_img_rgb)

# 9. Create separate images for each cluster (using original colors)
cluster_images = []
for i in range(k):
    cluster_img = np.zeros_like(img_rgb)
    cluster_mask = (full_labels == i)
    cluster_img[cluster_mask] = img_rgb[cluster_mask]
    cluster_images.append(Image.fromarray(cluster_img))

# 10. Display results: original image, composite segmentation, and each cluster
num_plots = 2 + k  # Original, composite, and each cluster image
plt.figure(figsize=(15, 5))

# Original image
plt.subplot(1, num_plots, 1)
plt.imshow(img_rgb)
plt.title("Original")
plt.axis("off")

# Composite segmented image (with centroid colors)
plt.subplot(1, num_plots, 2)
plt.imshow(segmented_pil)
plt.title("Segmented (Composite)")
plt.axis("off")

# Display each cluster separately
for i in range(k):
    plt.subplot(1, num_plots, 3 + i)
    plt.imshow(cluster_images[i])
    plt.title(f"Cluster {i}")
    plt.axis("off")

plt.tight_layout()
plt.show()


#Pre-process for the entire dataset

In [ ]:
import os
import cv2
import numpy as np
from PIL import Image
from sklearn.cluster import KMeans

# -------------------------
# Path Setup
# -------------------------
input_dir = "/content/drive/MyDrive/tomato_dataset/segmented/Tomato___Tomato_Yellow_Leaf_Curl_Virus"
output_dir = "/content/drive/MyDrive/tomato_dataset/kmean_clustered/Tomato___Tomato_Yellow_Leaf_Curl_Virus"
os.makedirs(output_dir, exist_ok=True)

# -------------------------
# Loop over each file
# -------------------------
for filename in os.listdir(input_dir):
    # Process only image files (jpg, jpeg, png, etc.)
    if filename.lower().endswith((".jpg", ".jpeg", ".png")):
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)  # same filename in new folder

        print(f"Processing: {input_path}")

        # 1. Load the image (BGR) and convert to RGB
        img_bgr = cv2.imread(input_path)
        if img_bgr is None:
            print(f"Warning: Could not read {input_path}. Skipping.")
            continue
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # 2. Convert to Lab color space
        img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)

        # 3. Create a mask to remove the black background
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        _, mask = cv2.threshold(gray, 5, 255, cv2.THRESH_BINARY)

        # 4. Extract only the leaf region from the Lab image using the mask.
        #    Use only the a and b channels (ignoring lightness) for clustering.
        lab_pixels = img_lab[mask > 0]
        ab_pixels = lab_pixels[:, 1:3].astype(np.float32)

        # 5. K-means clustering on a and b channels
        k = 5  # Change this value if you want more clusters.
        kmeans = KMeans(n_clusters=k, random_state=0, n_init=10)
        kmeans.fit(ab_pixels)
        cluster_labels = kmeans.labels_
        centers = kmeans.cluster_centers_

        # 6. Map the cluster labels back to the full image dimensions
        full_labels = np.full(mask.shape, -1, dtype=int)
        full_labels[mask > 0] = cluster_labels

        # 7. Create a composite image where each region is replaced with the average RGB color
        composite_avg_rgb = np.zeros_like(img_rgb)
        for i in range(k):
            cluster_mask = (full_labels == i)
            if np.any(cluster_mask):
                avg_color = np.mean(img_rgb[cluster_mask], axis=0)
                avg_color = np.array(avg_color, dtype=np.uint8)
                composite_avg_rgb[cluster_mask] = avg_color

        # 8. Save the composite image
        # Convert from RGB to BGR for cv2.imwrite.
        composite_bgr = cv2.cvtColor(composite_avg_rgb, cv2.COLOR_RGB2BGR)
        cv2.imwrite(output_path, composite_bgr)
        print(f"  --> Saved composite image to: {output_path}")

print("Batch processing complete.")


串流輸出內容已截斷至最後 5000 行。
  --> Saved composite image to: /content/drive/MyDrive/tomato_dataset/kmean_clustered/Tomato___Tomato_Yellow_Leaf_Curl_Virus/fc8c58be-8910-4fc4-a846-16baf3fbac5f___UF.GRC_YLCV_Lab 03128_final_masked.jpg
Processing: /content/drive/MyDrive/tomato_dataset/segmented/Tomato___Tomato_Yellow_Leaf_Curl_Virus/feece77e-16e6-48f9-86c6-80f6fa83c8b5___YLCV_GCREC 5477_final_masked.jpg
  --> Saved composite image to: /content/drive/MyDrive/tomato_dataset/kmean_clustered/Tomato___Tomato_Yellow_Leaf_Curl_Virus/feece77e-16e6-48f9-86c6-80f6fa83c8b5___YLCV_GCREC 5477_final_masked.jpg
Processing: /content/drive/MyDrive/tomato_dataset/segmented/Tomato___Tomato_Yellow_Leaf_Curl_Virus/9bf28103-50ef-4fdc-a06c-86b61d34e8ae___YLCV_NREC 2865_final_masked.jpg
  --> Saved composite image to: /content/drive/MyDrive/tomato_dataset/kmean_clustered/Tomato___Tomato_Yellow_Leaf_Curl_Virus/9bf28103-50ef-4fdc-a06c-86b61d34e8ae___YLCV_NREC 2865_final_masked.jpg
Processing: /content/drive/MyDrive/tomato_